# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rafayyk7/flyrank-ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

We build our model's feature vector using the stable, observable numeric columns from the starter dataset.

* **Feature List:** `content_age_days`, `days_since_last_update`, `impressions_90d`, `avg_position`, `ctr`, and `word_count`.
* **Categorical Handling:** No high-cardinality categorical features are used in this baseline setup.
* **Missing Values (Imputation):** Any missing values (`NaN`) in our features are filled with their respective column **medians** calculated from the training split. This protects against model evaluation crashes while avoiding the distortions that global means can introduce in skewed search console metrics (like impressions).

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import sys
import subprocess
import pandas as pd
import numpy as np

# 1. Clone repository in Google Colab to access the data directory
if 'google.colab' in sys.modules:
    REPO_URL = "https://github.com/rafayyk7/flyrank-ml.git"
    REPO_DIR = "flyrank-ml"

    if not os.path.exists(REPO_DIR):
        print("Cloning repository into Google Colab...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

    os.chdir(REPO_DIR)

# 2. Establish path to the starter dataset
csv_path = 'data/raw/content_refresh_anonymized.csv'
if not os.path.exists(csv_path):
    csv_path = '../../data/raw/content_refresh_anonymized.csv'

# 3. Load dataset and construct feature vector
df = pd.read_csv(csv_path)

# Extract features and label
features_list = ["content_age_days", "days_since_last_update", "impressions_90d", "avg_position", "ctr", "word_count"]
df['is_declining_label'] = df['trend_direction'].str.lower().eq('down').astype(int)

X = df[features_list].copy()
y = df['is_declining_label'].copy()

# Handle missing values by filling with column medians
print("--- Imputation Summary ---")
for col in features_list:
    missing_count = X[col].isnull().sum()
    if missing_count > 0:
        median_val = X[col].median()
        X[col] = X[col].fillna(median_val)
        print(f"Filled {missing_count:,} missing values in '{col}' with median: {median_val:.2f}")
    else:
        print(f"No missing values in '{col}'")

print("\n✅ Feature Matrix X and Target vector y successfully built!")

Cloning repository into Google Colab...
--- Imputation Summary ---
No missing values in 'content_age_days'
No missing values in 'days_since_last_update'
No missing values in 'impressions_90d'
No missing values in 'avg_position'
No missing values in 'ctr'
Filled 7,699 missing values in 'word_count' with median: 2877.00

✅ Feature Matrix X and Target vector y successfully built!


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Feature Name | Meaning | Missing Value Treatment | Available-When? (Temporal check) |
| :--- | :--- | :--- | :--- |
| `content_age_days` | Number of days since the URL was originally created. | Imputed with median. | Yes. Known at prediction time. |
| `days_since_last_update` | Number of days elapsed since the page was last updated. | Imputed with median. | Yes. Known at prediction time. |
| `impressions_90d` | Total organic search impressions over the last 90 days. | Imputed with median. | Yes. Known at prediction time (historical window). |
| `avg_position` | Average position in the Search Engine Result Pages (SERPs) over 90 days. | Imputed with median. | Yes. Known at prediction time (historical window). |
| `ctr` | Click-Through Rate ($Clicks/Impressions$) over the last 90 days. | Imputed with median. | Yes. Known at prediction time (historical window). |
| `word_count` | Total number of words currently present on the page. | Imputed with median. | Yes. Known at prediction time. |

*All selected features represent historical metrics up to Day 0. They are fully established before the subsequent 30-day forward-looking prediction window begins.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Print feature range check to verify sanity
print("--- Feature Validation & Bounds Check ---")
print(X.describe().loc[['mean', 'min', 'max']])

# Ensure no NaNs remain
assert not X.isnull().any().any(), "Error: There are still missing values in X!"
print("\n✅ Assertion passed: No missing values remain in the feature matrix.")

--- Feature Validation & Bounds Check ---
      content_age_days  days_since_last_update  impressions_90d  avg_position  \
mean          256.1678                 46.0983        5200.3663      16.34238   
min            90.0000                  1.0000           1.0000       0.00000   
max           564.0000                373.0000      517715.0000     245.00000   

             ctr   word_count  
mean    0.510733  3048.539533  
min     0.000000     8.000000  
max   100.000000  9546.000000  

✅ Assertion passed: No missing values remain in the feature matrix.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

### The Leakage Attack Plan:
Target leakage occurs if a feature accidentally previews the future or contains the target label in disguise. We will run a mathematical "leakage audit" by calculating the Pearson correlation coefficient ($r$) between every feature in $X$ and our target $y$.

If any feature shows an absolute correlation $|r| > 0.90$, it is flagged as highly suspicious of leaking the label.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("--- Launching Leakage Hunt ---")

# Compute correlation of each feature with the label y
correlations = X.apply(lambda col: col.corr(y))

leakage_found = False
for feature, corr_val in correlations.items():
    abs_corr = abs(corr_val)
    print(f"Correlation: '{feature}' with Target = {corr_val:.4f}")
    if abs_corr > 0.90:
        print(f"🚨 LEAKAGE WARNING: '{feature}' has an extremely high correlation ({corr_val:.4f}) with the target!")
        leakage_found = True

if not leakage_found:
    print("\n✅ Leakage Hunt Complete: No feature correlations exceed the 0.90 threshold. Feature set is clean!")

--- Launching Leakage Hunt ---
Correlation: 'content_age_days' with Target = -0.1639
Correlation: 'days_since_last_update' with Target = 0.0814
Correlation: 'impressions_90d' with Target = -0.0182
Correlation: 'avg_position' with Target = -0.0290
Correlation: 'ctr' with Target = -0.0619
Correlation: 'word_count' with Target = 0.0843

✅ Leakage Hunt Complete: No feature correlations exceed the 0.90 threshold. Feature set is clean!


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

We strictly refuse to include the following fields in our feature vector $X$:

1. **`trend_direction`**: Excluded because our target variable `is_declining_label` is derived directly from it (`trend_direction == 'down'`). Including it gives the model the exact answer.
2. **`trend_pct`**: Excluded because it represents the continuous percentage shift of the trend. It is mathematically linked to `trend_direction` and leaks the target.
3. **Downstream Product Flags (e.g., `health_score`, `needs_ctr_fix`)**: If these existed in our dataset, they would be excluded because they are computed downstream *after* a decline is detected. In production, these fields would not be populated at prediction time.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Programmatically assert that excluded columns are nowhere to be found in X
excluded_fields = ["trend_direction", "trend_pct", "is_declining_label", "label"]

print("--- Anti-Leakage Exclusion Check ---")
overlap = [field for field in excluded_fields if field in X.columns]

if len(overlap) == 0:
    print("✅ Verified: None of the target-derived or excluded columns exist in your training features!")
else:
    print(f"❌ LEAKAGE ERROR: The columns {overlap} were found in your training features!")

--- Anti-Leakage Exclusion Check ---
✅ Verified: None of the target-derived or excluded columns exist in your training features!


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.